# Kapitel 20.2 - Threading und Synchronisation

In diesem Notebook lernst du, wie Threads sicher zusammenarbeiten.

# Lernziele

- Race Conditions verstehen
- `Lock`, `RLock`, `Semaphore`, `Queue` anwenden
- Thread-sichere Muster einsetzen

# Voraussetzungen

- Kapitel 20.1
- Schleifen und Funktionen

# Theorie

Wenn mehrere Threads denselben Speicherbereich schreiben, kann es zu Race Conditions kommen.
Synchronisationswerkzeuge verhindern inkonsistente Zustande.

# Erklaerung

- `Lock`: ein Thread gleichzeitig
- `Semaphore`: begrenzte Anzahl gleichzeitig
- `Queue`: thread-sicherer Datenaustausch

# Syntax

```python
lock = threading.Lock()
with lock:
    # kritischer Bereich
    ...
```

# Merke

Ein funktionierendes Thread-Programm ist nicht automatisch korrekt.
Korrektheit braucht reproduzierbare Synchronisationsregeln.

# Parameter

- `Semaphore(value=n)`
- `Queue(maxsize=n)`
- `Thread(..., daemon=True|False)`

# Rueckgabewert

Thread-Ergebnisse werden oft ueber Queue oder gemeinsame Datenobjekte transportiert.

In [ ]:
# Beispiel 1: Unsicherer Zugriff
import threading

zaehler = 0

def inkrementiere_unsicher():
    global zaehler
    for _ in range(100000):
        zaehler += 1

threads = [threading.Thread(target=inkrementiere_unsicher) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print('Zaehler (unsicher):', zaehler)

In [ ]:
# Beispiel 2: Sicher mit Lock
zaehler = 0
lock = threading.Lock()

def inkrementiere_sicher():
    global zaehler
    for _ in range(100000):
        with lock:
            zaehler += 1

threads = [threading.Thread(target=inkrementiere_sicher) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print('Zaehler (sicher):', zaehler)

In [ ]:
# Beispiel 3: Producer-Consumer mit Queue
from queue import Queue

q = Queue()

def producer():
    for i in range(5):
        q.put(i)

def consumer():
    while not q.empty():
        print('Verarbeite', q.get())
        q.task_done()

producer()
consumer()

# Praxisbeispiel

Baue einen Download-Manager mit mehreren Worker-Threads und einer Queue.

# Haeufige Fehler

1. Lock vergessen bei gemeinsamem Zustand.
2. Deadlocks durch unguenstige Lock-Reihenfolge.
3. Queue ohne `task_done`/`join` verwenden.

# Best Practice

- Gemeinsam genutzte mutable Daten minimieren.
- Queue als Standard fuer Thread-Kommunikation bevorzugen.

# Tipp

Wenn moeglich, teile Arbeit in unabhaengige Jobs auf statt in gemeinsamen globalen Zustand.

# Uebung

Implementiere einen Worker-Pool mit 4 Threads und 20 Jobs.

# Loesung

Nutze `Queue`, starte 4 Worker-Threads, verarbeite Jobs in einer Schleife und verwende `queue.join()`.

# Zusammenfassung

Du kannst Thread-Synchronisation jetzt kontrolliert einsetzen und typische Risiken vermeiden.

# Weiterfuehrende Links

- Python `threading` docs
- Python `queue` docs

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

Race Condition
Critical Section
Event Loop
Backpressure
Cancellation
Timeout Budget

In [ ]:
# asyncio Timeout-Guard
import asyncio
async def slow_job():
    await asyncio.sleep(2)
    return "done"
async def main():
    try:
        print(await asyncio.wait_for(slow_job(), timeout=1.0))
    except asyncio.TimeoutError:
        print("timeout")
asyncio.run(main())

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.